In [5]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6960.3/8062.4 GB disk)


## Setup Dataset

In [8]:
import pandas as pd
path = "/kaggle/input/datasets/adilshamim8/rock-paper-scissors"
df_annotation_train = path + "/train/train/_annotations.csv"
df_annotation_train = pd.read_csv(df_annotation_train)
print(df_annotation_train)

                                               filename  width  height  \
0     egohands-public-1620914960773_png_jpg.rf.aa184...    640     640   
1     egohands-public-1624053434391_png_jpg.rf.aaef5...    640     640   
2     egohands-public-1624465902684_png_jpg.rf.aaa09...    640     640   
3     Screen-Shot-2022-02-08-at-12-59-24-PM_png.rf.a...    640     640   
4     egohands-public-1622127402076_png_jpg.rf.aa897...    640     640   
...                                                 ...    ...     ...   
4605  youtube-110_jpg.rf.617bbb713bb7283ec31ef55bc8c...    640     640   
4606  20220216_222247_jpg.rf.619399ee585a03994872dc3...    640     640   
4607  IMG_7043_MOV-93_jpg.rf.615ac5e9bc512fcc26af997...    640     640   
4608  egohands-public-1620849831752_png_jpg.rf.61844...    640     640   
4609  egohands-public-1624546380201_png_jpg.rf.61942...    640     640   

         class  xmin  ymin  xmax  ymax  
0         Rock   429   185   562   319  
1        Paper   269   354   

In [ ]:
import os

classes = sorted(df_annotation_train['class'].unique().tolist())
os.makedirs('/kaggle/working/train/labels', exist_ok=True)

def convert_yolo_format(row):
    class_id = classes.index(row['class'])
    x_center = (row['xmin'] + row['xmax']) / 2 / row['width']
    y_center = (row['ymin'] + row['ymax']) / 2 / row['height']
    bbox_width = (row['xmax'] - row['xmin']) / row['width']
    bbox_height = (row['ymax'] - row['ymin']) / row['height']
    return f"{class_id} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}"

for filename, group in df_annotation_train.groupby('filename'):
    yolo_labels = group.apply(convert_yolo_format, axis=1)
    from pathlib import Path
    label_path = f"/kaggle/working/train/labels/{Path(filename).stem}.txt"
    with open(label_path, "w") as f:
        f.write("\n".join(yolo_labels) + "\n")

In [ ]:
import shutil

src_dir = path+'/train/train'
dst_dir = "/kaggle/working/train/images"

os.makedirs(dst_dir, exist_ok=True)

# Copy images from source to destination
for filename in os.listdir(src_dir):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        shutil.copy(os.path.join(src_dir, filename), os.path.join(dst_dir, filename))

print(f"Copied {len(os.listdir(dst_dir))} images to {dst_dir}")

In [ ]:
yaml_content = """
train: /kaggle/working/train/images
val: /kaggle/working/train/images

nc: 3
names: ['Paper', 'Rock', 'Scissors']
"""
with open("data.yaml", "w") as f:
    f.write(yaml_content)

In [ ]:
labels_path = "/kaggle/working/train/labels"
txt_files = [f for f in os.listdir(labels_path) if f.endswith(".txt")]

## Fine Tuning

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo26n.pt")
model.train(data='data.yaml', epochs=5)
results = model.val()

In [ ]:
model.export(format="saved_model")